> **Step 1 — Environment Setup (Cell 1)**

In [14]:
# Install dependencies
!pip install timm einops -q

import os, math, random, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Multi-GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPUs available: {torch.cuda.device_count()}")

Using device: cuda
GPUs available: 1


> **Step 2 — Dataset Loading (Cell 2)**

In [4]:
BATCH_SIZE = 64
IMAGE_SIZE = 224
DATA_ROOT = os.path.join(akash2sharma_tiny_imagenet_path, "tiny-imagenet-200")

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 100000 | Val: 10000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


> **Step 3 — Patchification & Masking Utilities (Cell 3)**

In [19]:
def patchify(images, patch_size=16):
    """
    images: (B, 3, H, W)
    returns patches: (B, N, patch_size*patch_size*3)
    where N = (H/patch_size)*(W/patch_size)
    """
    B, C, H, W = images.shape
    assert H % patch_size == 0 and W % patch_size == 0
    h = H // patch_size  # 14
    w = W // patch_size  # 14
    # Reshape to patches
    x = images.reshape(B, C, h, patch_size, w, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)           # (B, h, w, C, ps, ps)
    x = x.reshape(B, h * w, C * patch_size * patch_size)  # (B, N, patch_dim)
    return x

def unpatchify(patches, patch_size=16, image_size=224):
    """
    patches: (B, N, patch_size*patch_size*3)
    returns images: (B, 3, H, W)
    """
    B, N, D = patches.shape
    h = w = image_size // patch_size
    C = 3
    x = patches.reshape(B, h, w, C, patch_size, patch_size)
    x = x.permute(0, 3, 1, 4, 2, 5)           # (B, C, h, ps, w, ps)
    x = x.reshape(B, C, h * patch_size, w * patch_size)
    return x

def random_masking(x, mask_ratio=0.75):
    """
    x: (B, N, D) — all patch tokens
    Returns:
        x_visible:   (B, n_visible, D)
        mask:        (B, N) — 1=masked, 0=visible
        ids_restore: (B, N) — indices to restore original order
    """
    B, N, D = x.shape
    n_keep = int(N * (1 - mask_ratio))   # 49 visible patches

    noise = torch.rand(B, N, device=x.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :n_keep]
    x_visible = torch.gather(x, dim=1,
                             index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

    mask = torch.ones(B, N, device=x.device)
    mask[:, :n_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)

    return x_visible, mask, ids_restore

**Step 4 — Transformer Building Blocks (Cell 4)**




In [20]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv   = nn.Linear(dim, dim * 3, bias=True)
        self.proj  = nn.Linear(dim, dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadSelfAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim    = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

**Step 5 — MAE Encoder (ViT-Base) (Cell 5)**

In [ ]:
class MAEEncoder(nn.Module):
    """
    ViT-Base/16 encoder.
    Only processes VISIBLE patches (25%).
    """
    def __init__(self,
                 image_size=224, patch_size=16,
                 in_chans=3,   embed_dim=768,
                 depth=12,     num_heads=12):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2  # 196

        # Patch embedding (linear projection)
        patch_dim = in_chans * patch_size * patch_size
        self.patch_embed = nn.Linear(patch_dim, embed_dim)

        # [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Positional embeddings (196 patches + 1 CLS)
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim), requires_grad=False
        )
        self._init_pos_embed()

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        self._init_weights()

    def _init_pos_embed(self):
        """Sinusoidal positional embeddings."""
        pos_embed = self._get_sinusoid_pos_embed(
            self.num_patches + 1, self.pos_embed.shape[-1]
        )
        self.pos_embed.data.copy_(pos_embed.float().unsqueeze(0))

    @staticmethod
    def _get_sinusoid_pos_embed(n_pos, d_model):
        position = torch.arange(n_pos).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(n_pos, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def _init_weights(self):
        nn.init.normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x_visible, ids_keep):
        """
        x_visible: (B, n_keep, patch_dim) — raw visible patches
        ids_keep:  (B, n_keep)            — original patch indices
        Returns latent: (B, n_keep, embed_dim)  [without CLS]
        """
        B, n_keep, _ = x_visible.shape

        # Project patches to embed dim
        x = self.patch_embed(x_visible)   # (B, n_keep, embed_dim)

        # Add positional embeddings (skip CLS at index 0)
        pos = self.pos_embed[:, 1:, :]    # (1, 196, embed_dim)
        pos_visible = torch.gather(
            pos.expand(B, -1, -1), 1,
            ids_keep.unsqueeze(-1).expand(-1, -1, pos.shape[-1])
        )
        x = x + pos_visible

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)  # (B, n_keep+1, embed_dim)

        # Transformer layers
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)

        return x[:, 1:, :]  # Remove CLS, return (B, n_keep, embed_dim)

**Step 6 — MAE Decoder (ViT-Small) (Cell 6)**

In [21]:
class MAEDecoder(nn.Module):
    """
    Lightweight ViT-Small/16 decoder.
    Reconstructs ALL 196 patches from 49 visible + 147 mask tokens.
    """
    def __init__(self,
                 num_patches=196, patch_size=16,
                 encoder_dim=768, decoder_dim=384,
                 depth=12,        num_heads=6,
                 in_chans=3):
        super().__init__()
        patch_dim = in_chans * patch_size * patch_size

        # Project encoder dim -> decoder dim
        self.decoder_embed = nn.Linear(encoder_dim, decoder_dim)

        # Learnable mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))

        # Decoder positional embeddings
        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, decoder_dim), requires_grad=False
        )
        self._init_pos_embed(num_patches)

        self.blocks = nn.ModuleList([
            TransformerBlock(decoder_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(decoder_dim)

        # Prediction head: decoder_dim -> patch pixels
        self.pred = nn.Linear(decoder_dim, patch_dim)

        self._init_weights()

    def _init_pos_embed(self, num_patches):
        pos_embed = MAEEncoder._get_sinusoid_pos_embed(
            num_patches + 1, self.decoder_pos_embed.shape[-1]
        )
        self.decoder_pos_embed.data.copy_(pos_embed.float().unsqueeze(0))

    def _init_weights(self):
        nn.init.normal_(self.mask_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, latent, ids_restore):
        """
        latent:      (B, n_keep, encoder_dim)
        ids_restore: (B, N)
        Returns pred: (B, N, patch_dim)
        """
        B = latent.shape[0]
        N = ids_restore.shape[1]          # 196 total patches
        n_keep = latent.shape[1]           # 49 visible

        # Project to decoder dim
        x = self.decoder_embed(latent)    # (B, n_keep, decoder_dim)

        # Create mask tokens for missing positions
        mask_tokens = self.mask_token.expand(B, N - n_keep, -1)

        # Concatenate and unshuffle to original order
        x_full = torch.cat([x, mask_tokens], dim=1)  # (B, N, decoder_dim)
        x_full = torch.gather(
            x_full, 1,
            ids_restore.unsqueeze(-1).expand(-1, -1, x_full.shape[-1])
        )

        # Add positional embeddings (skip CLS)
        x_full = x_full + self.decoder_pos_embed[:, 1:, :]

        # Transformer layers
        for blk in self.blocks:
            x_full = blk(x_full)
        x_full = self.norm(x_full)

        # Pixel prediction
        pred = self.pred(x_full)          # (B, N, patch_dim)
        return pred

**Step 7 — Full MAE Model + Loss (Cell 7)**

In [ ]:
class MAE(nn.Module):
    def __init__(self, image_size=224, patch_size=16, mask_ratio=0.75):
        super().__init__()
        self.mask_ratio  = mask_ratio
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2

        self.encoder = MAEEncoder(image_size=image_size, patch_size=patch_size)
        self.decoder = MAEDecoder(num_patches=self.num_patches, patch_size=patch_size)

    def forward(self, images):
        """
        images: (B, 3, 224, 224)
        Returns: loss (scalar), pred (B,N,D), mask (B,N)
        """
        # Step 1: Patchify
        patches = patchify(images, self.patch_size)      # (B, 196, 768)

        # Step 2: Random masking
        B, N, D = patches.shape
        n_keep  = int(N * (1 - self.mask_ratio))

        noise        = torch.rand(B, N, device=images.device)
        ids_shuffle  = torch.argsort(noise, dim=1)
        ids_restore  = torch.argsort(ids_shuffle, dim=1)
        ids_keep     = ids_shuffle[:, :n_keep]

        x_visible = torch.gather(
            patches, 1,
            ids_keep.unsqueeze(-1).expand(-1, -1, D)
        )
        mask = torch.ones(B, N, device=images.device)
        mask[:, :n_keep] = 0
        mask = torch.gather(mask, 1, ids_restore)

        # Step 3: Encode visible patches
        latent = self.encoder(x_visible, ids_keep)       # (B, n_keep, 768)

        # Step 4: Decode
        pred = self.decoder(latent, ids_restore)          # (B, 196, 768)

        # Step 5: MSE loss on masked patches ONLY
        loss = self.compute_loss(patches, pred, mask)

        return loss, pred, mask

    def compute_loss(self, target, pred, mask):
        """MSE only on masked patches."""
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)          # (B, N)
        loss = (loss * mask).sum() / mask.sum()
        return loss

**Step 8 — Training Loop (Cell 8)**

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Model ──────────────────────────────────────────
model = MAE(image_size=224, patch_size=16, mask_ratio=0.75)

# Multi-GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
    model = nn.DataParallel(model)

model = model.to(device)

# ── Optimizer & Scheduler ──────────────────────────
EPOCHS    = 50
LR        = 1.5e-4
WD        = 0.05

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler()   # Mixed precision

# ── Training ───────────────────────────────────────
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches  = 0

    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():  # Mixed precision
            loss, pred, mask = model(images)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_batches  += 1

        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Batch [{batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    scheduler.step()

    print(f"\n✓ Epoch {epoch+1}/{EPOCHS} — Avg Loss: {avg_loss:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}\n")

# ── Save checkpoint ────────────────────────────────
torch.save(model.state_dict(), "mae_checkpoint.pth")
print("Checkpoint saved!")